# 300m Park Access — Statistics & Filter Justification

This notebook loads the final 300m computed outputs and:
1. explore the 300m access statistics
2. Justifies the **>6 buildings** threshold used to filter `ar_vrd` polygons (private courtyards / block interiors)

In [1]:
import os
import geopandas as gpd
import pandas as pd
import sqlite3

os.chdir("/Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index")

buildings_routed = gpd.read_file('data/300M_indicator.gpkg', driver='GPKG', layer='buildings')
parks            = gpd.read_file('data/300M_indicator.gpkg', driver='GPKG', layer='parks')
parks_full = pd.concat([
    gpd.read_file('data/greenery_all.gpkg', layer=l)
    for l in ['ar_vrd_all', 'pe_unis_all', 'osm_all_public']
], ignore_index=True)


/opt/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(
/opt/anaconda3/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


## 1 · Headline access statistics

In [2]:
total   = len(buildings_routed)
within  = buildings_routed['within_300m'].sum()
outside = total - within

print('300m PARK ACCESS INDICATOR — BOLOGNA')

print(f'\n── ADDRESS POINTS ──')
print(f'  Total civici:                {total:>8,}')
print(f'  Within 300m walk of park:    {within:>8,}  ({within/total*100:.1f}%)')
print(f'  Outside 300m:                {outside:>8,}  ({outside/total*100:.1f}%)')
print(f'  No park reachable (NaN):     {buildings_routed["park_id"].isna().sum():>8,}')

print(f'\n── PARKS ──')
print(f'  Total parks (filtered):      {len(parks):>8,}')
print(f'  by layer type:')
print(parks_full['layer_type'].value_counts().to_string())


300m PARK ACCESS INDICATOR — BOLOGNA

── ADDRESS POINTS ──
  Total civici:                  77,630
  Within 300m walk of park:      53,442  (68.8%)
  Outside 300m:                  24,188  (31.2%)
  No park reachable (NaN):       24,188

── PARKS ──
  Total parks (filtered):           698
  by layer type:
layer_type
ar_vrd_all         3814
park_garden_all    1030
pe_unis_all         542


## 2 · `ar_vrd` filter justification — why threshold = 6 buildings

The `ar_vrd_all` layer includes **courtyards and private block interiors** that appear as green polygons but are not publicly accessible. These are identified by counting how many building footprints intersect each polygon — a high count indicates an enclosed residential block, not a public park.

The threshold of **≤ 6 buildings** was chosen by inspecting the distribution below and the sensitivity table.

In [4]:
from shapely.strtree import STRtree

CRS_METRIC = 'EPSG:32632'

building_footprints = (
    gpd.read_file('data/bologna_buildings_final.gpkg')
    .to_crs(CRS_METRIC)
    .loc[lambda df: df.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
)
footprint_tree = STRtree(building_footprints.geometry.values)

ar_vrd = (
    gpd.read_file('data/300m/greenery_all.gpkg', layer='ar_vrd_all')
    .to_crs(CRS_METRIC)
)
ar_vrd['n_buildings_inside'] = ar_vrd.geometry.apply(
    lambda g: len(footprint_tree.query(g, predicate='intersects'))
)

print('Distribution of buildings-inside count for ar_vrd polygons:')
print(ar_vrd['n_buildings_inside'].describe().round(2))


Distribution of buildings-inside count for ar_vrd polygons:
count    3814.00
mean       10.50
std        16.95
min         0.00
25%         1.00
50%         4.00
75%        14.00
max       201.00
Name: n_buildings_inside, dtype: float64


In [5]:
print('Sensitivity of threshold choice:')
print(f'  {"threshold":<15} {"kept":>6} {"removed":>8} {"% removed":>10}')
print('  ' + '-'*42)
for thresh in [2, 4, 6, 8, 10, 15]:
    kept    = (ar_vrd['n_buildings_inside'] <= thresh).sum()
    removed = len(ar_vrd) - kept
    marker  = '  ← chosen' if thresh == 6 else ''
    print(f'  <= {thresh:<12} {kept:>6,} {removed:>8,} {removed/len(ar_vrd)*100:>9.1f}%{marker}')

ar_vrd_kept    = (parks['layer_type'] == 'ar_vrd_all').sum() if 'layer_type' in parks.columns else 'n/a'
ar_vrd_removed = len(ar_vrd) - ar_vrd_kept
print(f'\nar_vrd before filter: {len(ar_vrd):,}')
print(f'ar_vrd after  filter: {ar_vrd_kept:,}  (removed {ar_vrd_removed:,} polygons enclosing >6 footprints)')


Sensitivity of threshold choice:
  threshold         kept  removed  % removed
  ------------------------------------------
  <= 2             1,599    2,215      58.1%
  <= 4             1,958    1,856      48.7%
  <= 6             2,235    1,579      41.4%  ← chosen
  <= 8             2,453    1,361      35.7%
  <= 10            2,616    1,198      31.4%
  <= 15            2,974      840      22.0%

ar_vrd before filter: 3,814
ar_vrd after  filter: 481  (removed 3,333 polygons enclosing >6 footprints)


## 3 · Park size distribution (reachable parks)

In [6]:
if 'park_area_ha' in buildings_routed.columns:
    sizes = buildings_routed.dropna(subset=['park_area_ha'])['park_area_ha']
    print('Park area (ha) — buildings matched to a reachable park:')
    print(f'  Mean:   {sizes.mean():.2f} ha')
    print(f'  Median: {sizes.median():.2f} ha')
    print(f'  Min:    {sizes.min():.2f} ha')
    print(f'  Max:    {sizes.max():.2f} ha')

if 'park_type' in buildings_routed.columns:
    print('\nPark type breakdown (buildings within 300m):')
    print(buildings_routed[buildings_routed['within_300m']]['park_type'].value_counts().to_string())


Park area (ha) — buildings matched to a reachable park:
  Mean:   1.01 ha
  Median: 0.24 ha
  Min:    0.00 ha
  Max:    45.60 ha

Park type breakdown (buildings within 300m):
park_type
ar_vrd_all         39431
park_garden_all    10048
pe_unis_all         3963
